# Cluster im MFA-Raum (3D), getrennt für Discovery und Validation

Zeigt für jede Stichprobe die Clustermittelpunkte und die Streuung der Personen (Ellipsoide) in den ersten drei MFA-Dimensionen.

**Wichtig:** Die MFA wurde für Discovery und Validation **getrennt** gerechnet. Jede Stichprobe hat daher ihr eigenes Koordinatensystem. Die beiden Abbildungen zeigen, wie die Cluster *innerhalb* einer Stichprobe zueinander liegen. Positionen zwischen den Abbildungen dürfen nicht verglichen werden (Achsen können gespiegelt, gedreht oder anders skaliert sein). Die Replikation wird über die Antwortprofile gezeigt (Notebook `cluster_replication`).

## Schritt 0: Daten laden
MFA-Koordinaten, GMM-Labels und Surveygewichte. Ausgegeben wird, wie viel der Varianz aller gespeicherten MFA-Dimensionen auf die ersten drei entfällt – das gehört in die Bildunterschrift.

In [ ]:
# ------------------------------------------------------------
# Daten: MFA-Koordinaten (wie im GMM-Notebook), Clusterlabels, Surveygewichte
# ------------------------------------------------------------
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

project_path = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
os.chdir(project_path)
sys.path.insert(0, str(project_path))

from utils.saving_utils import DatasetBundle

output_folder = Path("outputs/cluster_mfa_3d_getrennt")
output_folder.mkdir(parents=True, exist_ok=True)

mfa_D = pd.read_parquet("data/dim_red/data_discov_mfa.parquet").reset_index(drop=True)
mfa_V = pd.read_parquet("data/dim_red/data_valid_mfa.parquet").reset_index(drop=True)
lab_D = pd.read_parquet("outputs/gmm_clustering/cluster_pred_discov.parquet")["label"].astype(str).to_numpy()
lab_V = pd.read_parquet("outputs/gmm_clustering/cluster_pred_valid.parquet")["label"].astype(str).to_numpy()
w_D = pd.to_numeric(DatasetBundle.load(Path("data/preprocessed/data_discov_full")).data["wichtn"]).to_numpy(dtype=float)
w_V = pd.to_numeric(DatasetBundle.load(Path("data/preprocessed/data_valid_full")).data["wichtn"]).to_numpy(dtype=float)
assert len(mfa_D) == len(lab_D) == len(w_D) and len(mfa_V) == len(lab_V) == len(w_V)

DIMS = list(mfa_D.columns[:3])     # die ersten drei MFA-Dimensionen
print("Verwendete Dimensionen:", DIMS)

# Anteil der ersten drei Dimensionen an der Gesamtvarianz aller gespeicherten MFA-Dimensionen
# (für die Bildunterschrift; Discovery und Validation haben eigene MFA-Räume)
for name, mfa in [("Discovery", mfa_D), ("Validation", mfa_V)]:
    var = mfa.var()
    share = var[DIMS] / var.sum()
    print(f"{name}: " + ", ".join(f"{d}: {s:.1%}" for d, s in share.items()) + f"  | zusammen {share.sum():.1%}")

## Schritt 1: Mittelpunkt und Streuung je Cluster
Gewichteter Mittelpunkt und gewichtete Kovarianz der Personen jedes Clusters in den ersten drei Dimensionen.

In [ ]:
# ------------------------------------------------------------
# Mittelpunkt und Streuung (Kovarianz) jedes Clusters in den drei Dimensionen, gewichtet
# ------------------------------------------------------------
def sort_clusters(cs):
    return sorted(cs, key=lambda c: int(c) if str(c).isdigit() else str(c))

def cluster_summary(mfa, labels, w):
    X = mfa[DIMS].to_numpy(dtype=float)
    out = {}
    for c in sort_clusters(np.unique(labels)):
        m = labels == c
        wc = w[m] / w[m].sum()
        mu = wc @ X[m]
        diff = X[m] - mu
        cov = (wc[:, None] * diff).T @ diff
        out[c] = {"mu": mu, "cov": cov, "n": int(m.sum()), "anteil": w[m].sum() / w.sum()}
    return out

sum_D = cluster_summary(mfa_D, lab_D, w_D)
sum_V = cluster_summary(mfa_V, lab_V, w_V)

# Hinweis: Discovery und Validation wurden getrennt per MFA transformiert.
# Die Koordinaten sind daher NUR innerhalb einer Stichprobe vergleichbar.
centroids = pd.concat([
    pd.DataFrame({c: s["mu"] for c, s in sum_D.items()}, index=DIMS).T.assign(stichprobe="D", n=[s["n"] for s in sum_D.values()]),
    pd.DataFrame({c: s["mu"] for c, s in sum_V.items()}, index=DIMS).T.assign(stichprobe="V", n=[s["n"] for s in sum_V.values()]),
])
print("Clustermittelpunkte in den ersten drei MFA-Dimensionen:")
display(centroids.round(3))
centroids.to_csv(output_folder / "mittelpunkte_mfa_3d.csv")

## Schritt 2: Einstellungen und Zeichenfunktion
- `SCALE`: Größe der Ellipsoide (1 = eine Standardabweichung, 2.8 ≈ 95 % der Personen)
- `MIN_N`: Ellipsoide nur für Cluster ab dieser Größe; kleinere Cluster erscheinen nur als Punkt
- `VIEWS`: drei Blickwinkel, weil ein 3D-Bild auf einer Folie je nach Perspektive täuscht
- Farben: feste Reihenfolge nach Clustergröße; die 8 größten Cluster bekommen eine eigene Farbe, alle weiteren grau. Jeder Punkt ist zusätzlich beschriftet.

Tipp: Mit `%matplotlib widget` (Paket `ipympl`) lässt sich der Plot mit der Maus drehen.

In [ ]:
# ------------------------------------------------------------
# Einstellungen und Zeichenfunktion
# ------------------------------------------------------------
SCALE = 1.0          # Ellipsoid-Größe: 1 = eine Standardabweichung, 2.8 = ca. 95 % der Personen
MIN_N = 100          # Ellipsoide nur für Cluster mit mindestens so vielen Personen; kleinere nur als Punkt
VIEWS = [(20, -60), (20, 30), (70, -90)]   # Blickwinkel (Höhe, Drehung)

# Feste Farbreihenfolge; die 8 größten Cluster bekommen eine Farbe, alle weiteren grau
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
GREY = "#9a9a9a"

def ellipsoid_mesh(mu, cov, scale=1.0, n=24):
    vals, vecs = np.linalg.eigh(cov)
    radii = scale * np.sqrt(np.clip(vals, 0, None))
    u = np.linspace(0, 2 * np.pi, n)
    v = np.linspace(0, np.pi, n // 2)
    sphere = np.stack([np.outer(np.cos(u), np.sin(v)),
                       np.outer(np.sin(u), np.sin(v)),
                       np.outer(np.ones_like(u), np.cos(v))])
    pts = np.einsum("ij,jkl->ikl", vecs * radii, sphere)
    return pts[0] + mu[0], pts[1] + mu[1], pts[2] + mu[2]


def plot_clusters_3d(summ, prefix, title, file_name):
    by_size = sorted(summ, key=lambda c: summ[c]["n"], reverse=True)
    colors = {c: (PALETTE[i] if i < len(PALETTE) else GREY) for i, c in enumerate(by_size)}

    fig = plt.figure(figsize=(6 * len(VIEWS), 6.5))
    for i, (elev, azim) in enumerate(VIEWS, 1):
        ax = fig.add_subplot(1, len(VIEWS), i, projection="3d")
        for c in by_size:
            s, col = summ[c], colors[c]
            if s["n"] >= MIN_N:
                x, y, z = ellipsoid_mesh(s["mu"], s["cov"], SCALE)
                ax.plot_wireframe(x, y, z, color=col, linewidth=0.3, alpha=0.4)
            ax.scatter(*s["mu"], s=30 + 1200 * s["anteil"], c=col, edgecolors="white",
                       linewidths=0.8, depthshade=False)
            ax.text(*s["mu"], f" {prefix}{c}", fontsize=7, color="#222222")
        ax.set_xlabel(DIMS[0]); ax.set_ylabel(DIMS[1]); ax.set_zlabel(DIMS[2])
        ax.tick_params(labelsize=7)
        ax.view_init(elev=elev, azim=azim)
        ax.set_title(f"Ansicht {i}", fontsize=9)

    handles = [plt.Line2D([], [], marker="o", ls="", color=colors[c],
                          label=f"{prefix}{c}  (n = {summ[c]['n']}, {summ[c]['anteil']:.1%})") for c in by_size]
    fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, -0.08), ncol=6, frameon=False, fontsize=8)
    fig.suptitle(f"{title}\nClustermittelpunkte (Größe ~ gewichteter Anteil) und Streuung ({SCALE:g} SD) "
                 f"in den ersten drei MFA-Dimensionen; Ellipsoide für n ≥ {MIN_N}", y=1.03)
    plt.tight_layout()
    plt.savefig(output_folder / f"{file_name}.png", dpi=300, bbox_inches="tight")
    plt.show()

## Schritt 3: Discovery

In [ ]:
plot_clusters_3d(sum_D, "D", "Discovery", "discovery_cluster_3d")

## Schritt 4: Validation

In [ ]:
plot_clusters_3d(sum_V, "V", "Validation", "validation_cluster_3d")